In [2]:
import pandas as pd
import sqlite3
import os
import logging

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append("/content/drive/MyDrive/inventory_project")

# The external ingestion_db.py file was previously identified as malformed (JSON).
# To ensure the script runs, a local ingest_db function is provided here.
# If you fix your external ingestion_db.py, you can remove this local definition
# and uncomment `from ingestion_db import ingest_db`.

BASE_PROJECT = "/content/drive/MyDrive/inventory_project"
DB_PATH = os.path.join(BASE_PROJECT, "inventory.db")
LOG_PATH = os.path.join(BASE_PROJECT, "logs") # Define a directory for logs
LOG_FILE_PATH = os.path.join(LOG_PATH, "get_vendor_summary.log")

os.makedirs(LOG_PATH, exist_ok=True) # Create the log directory if it doesn't exist

# logging.basicConfig(
#     filename=LOG_FILE_PATH,
#     level=logging.DEBUG,
#     format= "%(asctime)s | %(levelname)s | %(message)s",
#     filemode = "a"
# )

# Reset logging (important in Colab)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE_PATH),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)

def ingest_db(df, table_name):
  """Ingests a pandas DataFrame into a SQLite database table."""
  conn = None

  try:
    conn = sqlite3.connect(DB_PATH)

    df.to_sql(
        table_name,
        conn,
        if_exists='replace',
        index=False
        )

    logging.info(f"DataFrame successfully ingested into table '{table_name}'.")

  except Exception as e:
    logging.error(f"Error in ingesting DataFrame into table '{table_name}': {e}")

  finally:
    if conn:
      conn.close()

# def ingest_db(df, table_name):
#     df.to_sql(
#         table_name,
#         con = engine,
#         if_exists = "replace",
#         index = False,
#         chunksize=50_000
#     )

def create_vendor_summary(conn):
 vendor_sales_summary = pd.read_sql("""WITH FreightSummary AS (
        SELECT
            vendornumber,
            SUM(freight) AS FreightCost
        FROM vendor_invoice
        GROUP BY vendornumber
    ),
    PurchaseSummary AS (
        SELECT
            p.vendornumber,
            p.vendorname,
            p.brand,
            p.description,
            p.purchaseprice,
            pp.price AS ActualPrice,
            pp.volume,
            SUM(p.quantity) AS TotalPurchaseQuantity,
            SUM(p.dollars) AS TotalPurchaseDollars
        FROM purchases p
        JOIN purchase_prices pp
            ON p.brand = pp.brand
        WHERE p.purchaseprice > 0
        GROUP BY
            p.vendornumber, p.vendorname, p.brand,
            p.description, p.purchaseprice, pp.price, pp.volume
    ),
    SalesSummary AS (
        SELECT
            vendorno,
            brand,
            SUM(salesquantity) AS TotalSalesQuantity,
            SUM(salesdollars) AS TotalSalesDollars,
            SUM(salesprice) AS TotalSalesPrice,
            SUM(excisetax) AS TotalExciseTax
        FROM sales
        GROUP BY vendorno, brand
    )
    SELECT
        ps.vendornumber,
        ps.vendorname,
        ps.brand,
        ps.description,
        ps.purchaseprice,
        ps.ActualPrice,
        ps.volume,
        ps.TotalPurchaseQuantity,
        ps.TotalPurchaseDollars,
        ss.TotalSalesQuantity,
        ss.TotalSalesDollars,
        ss.TotalSalesPrice,
        ss.TotalExciseTax,
        fs.FreightCost
    FROM PurchaseSummary ps
    LEFT JOIN SalesSummary ss
        ON ps.vendornumber = ss.vendorno
        AND ps.brand = ss.brand
    LEFT JOIN FreightSummary fs
        ON ps.vendornumber = fs.vendornumber
    ORDER BY ps.TotalPurchaseDollars DESC""", conn)

 return vendor_sales_summary


def clean_data(df):
  '''this function will clean the data'''
  df['volume'] = df['volume'].astype(float)

  df.fillna(0, inplace=True)

  df['vendorname'] = df['vendorname'].str.strip()
  df['description'] = df['description'].str.strip()

  # Corrected DataFrame column assignments
  df['GrossProfit'] = df['TotalSalesDollars'] - df['TotalPurchaseDollars']
  df['ProfitMargin'] = (df['GrossProfit'] / df['TotalSalesDollars'])*100
  df['StockTurnover'] = df['TotalSalesQuantity'] / df['TotalPurchaseQuantity']
  df['SalesPurchasesRatio'] = df['TotalSalesDollars'] / df['TotalPurchaseDollars']

  return df

if __name__ == "__main__":

  conn = sqlite3.connect(DB_PATH) # Use DB_PATH for consistency

  logging.info('Creating Vendor Summary Table.....')
  summary_df = create_vendor_summary(conn)
  logging.info(summary_df.head())

  logging.info('Cleaning Data.....')
  clean_df = clean_data(summary_df)
  logging.info(clean_df.head())

  logging.info('Ingesting Data.....')
  ingest_db(clean_df, 'vendor_sales_summary')

  logging.info('Closing Connection.....') # Corrected logging call
  conn.close() # Ensure the connection opened in __main__ is closed
  logging.info('Completed')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


2026-05-12 07:59:01,678 | INFO | Creating Vendor Summary Table.....
2026-05-12 08:00:17,339 | INFO |    vendornumber                   vendorname  brand              description  \
0          1128  BROWN-FORMAN CORP             1233  Jack Daniels No 7 Black   
1          4425        MARTIGNETTI COMPANIES   3405    Tito's Handmade Vodka   
2         17035  PERNOD RICARD USA             8068         Absolut 80 Proof   
3          3960  DIAGEO NORTH AMERICA INC      4261   Capt Morgan Spiced Rum   
4          3960  DIAGEO NORTH AMERICA INC      3545          Ketel One Vodka   

   purchaseprice  ActualPrice volume  TotalPurchaseQuantity  \
0          26.27        36.99   1750                 145080   
1          23.19        28.99   1750                 164038   
2          18.24        24.99   1750                 187407   
3          16.17        22.99   1750                 201682   
4          21.89        29.99   1750                 138109   

   TotalPurchaseDollars  TotalSalesQuan

In [3]:
# import pandas as pd
# import sqlite3
# import os
# import logging
# import sys

# from google.colab import drive
# drive.mount('/content/drive')
# sys.path.append("/content/drive/MyDrive/inventory_project")
# from ingestion_db import ingest_db

# BASE_PROJECT = "/content/drive/MyDrive/inventory_project"
# DB_PATH = os.path.join(BASE_PROJECT, "inventory.db")
# LOG_DIR = os.path.join(BASE_PROJECT, "logs")
# LOG_FILE = os.path.join(LOG_DIR, "get_vendor_summary.log")

# os.makedirs(LOG_DIR, exist_ok=True)

# # Reset logging (important in Colab)
# for handler in logging.root.handlers[:]:
#     logging.root.removeHandler(handler)

# logging.basicConfig(
#     level=logging.INFO,
#     format="%(asctime)s | %(levelname)s | %(message)s",
#     handlers=[
#         logging.FileHandler(LOG_FILE),
#         logging.StreamHandler()
#     ]
# )

# logger = logging.getLogger(__name__)

# def create_vendor_summary(conn):
#     query = """
#     WITH FreightSummary AS (
#         SELECT vendornumber, SUM(freight) AS FreightCost
#         FROM vendor_invoice
#         GROUP BY vendornumber
#     ),
#     PurchaseSummary AS (
#         SELECT
#             p.vendornumber,
#             p.vendorname,
#             p.brand,
#             p.description,
#             p.purchaseprice,
#             pp.price AS ActualPrice,
#             pp.volume,
#             SUM(p.quantity) AS TotalPurchaseQuantity,
#             SUM(p.dollars) AS TotalPurchaseDollars
#         FROM purchases p
#         JOIN purchase_prices pp
#             ON p.brand = pp.brand
#         WHERE p.purchaseprice > 0
#         GROUP BY
#             p.vendornumber, p.vendorname, p.brand,
#             p.description, p.purchaseprice, pp.price, pp.volume
#     ),
#     SalesSummary AS (
#         SELECT
#             vendorno,
#             brand,
#             SUM(salesquantity) AS TotalSalesQuantity,
#             SUM(salesdollars) AS TotalSalesDollars,
#             SUM(excisetax) AS TotalExciseTax
#         FROM sales
#         GROUP BY vendorno, brand
#     )
#     SELECT
#         ps.vendornumber,
#         ps.vendorname,
#         ps.brand,
#         ps.description,
#         ps.purchaseprice,
#         ps.ActualPrice,
#         ps.volume,
#         ps.TotalPurchaseQuantity,
#         ps.TotalPurchaseDollars,
#         ss.TotalSalesQuantity,
#         ss.TotalSalesDollars,
#         ss.TotalExciseTax,
#         fs.FreightCost
#     FROM PurchaseSummary ps
#     LEFT JOIN SalesSummary ss
#         ON ps.vendornumber = ss.vendorno
#         AND ps.brand = ss.brand
#     LEFT JOIN FreightSummary fs
#         ON ps.vendornumber = fs.vendornumber
#     ORDER BY ps.TotalPurchaseDollars DESC
#     """
#     return pd.read_sql(query, conn)

# def clean_data(df):
#     df['volume'] = df['volume'].astype(float)
#     df.fillna(0, inplace=True)

#     df['vendorname'] = df['vendorname'].str.strip()
#     df['description'] = df['description'].str.strip()

#     df['GrossProfit'] = df['TotalSalesDollars'] - df['TotalPurchaseDollars']
#     df['ProfitMargin'] = (df['GrossProfit'] / df['TotalPurchaseDollars']) * 100
#     df['StockTurnover'] = df['TotalSalesQuantity'] / df['TotalPurchaseQuantity']
#     df['SalesPurchasesRatio'] = df['TotalSalesDollars'] / df['TotalPurchaseDollars']

#     return df

# if __name__ == "__main__":

#     logger.info("Connecting to database...")
#     conn = sqlite3.connect(DB_PATH)

#     try:
#         logger.info("Creating vendor summary...")
#         summary_df = create_vendor_summary(conn)
#         logger.info(summary_df.head())

#         logger.info("Cleaning data...")
#         clean_df = clean_data(summary_df)
#         logger.info(clean_df.head())

#         logger.info("Ingesting data into database...")
#         ingest_db(clean_df, "vendor_sales_summary")

#         logger.info("Process completed successfully")

#     except Exception as e:
#         logger.exception("Error occurred during vendor summary creation")

#     finally:
#         conn.close()
#         logger.info("Database connection closed")


In [4]:
# import pandas as pd
# import sqlite3
# import os
# import logging

# # from google.colab import drive
# # drive.mount('/content/drive')

# import sys
# sys.path.append("/content/drive/MyDrive/inventory_project")

# from ingestion_db import ingest_db


# BASE_PROJECT = "/content/drive/MyDrive/inventory_project"
# DB_PATH = os.path.join(BASE_PROJECT, "inventory.db")
# LOG_PATH = os.path.join(BASE_PROJECT, "logs", "get_vendor_summary.log")

# # os.makedirs(LOG_PATH, exist_ok=True)

# logging.basicConfig(
#     filename=os.path.join(LOG_PATH, "get_vendor_summary.log"),
#     level=logging.DEBUG,
#     format= "%(asctime)s | %(levelname)s | %(message)s",
#     filemode = "a"
# )

# def create_vendor_summary(conn):
#  vendor_sales_summary = pd.read_sql("""WITH FreightSummary AS (
#         SELECT vendornumber, SUM(freight) AS FreightCost
#         FROM vendor_invoice
#         GROUP BY vendornumber
#     ),
#     PurchaseSummary AS (
#         SELECT
#             p.vendornumber,
#             p.vendorname,
#             p.brand,
#             p.description,
#             p.purchaseprice,
#             pp.price AS ActualPrice,
#             pp.volume,
#             SUM(p.quantity) AS TotalPurchaseQuantity,
#             SUM(p.dollars) AS TotalPurchaseDollars
#         FROM purchases p
#         JOIN purchase_prices pp
#             ON p.brand = pp.brand
#         WHERE p.purchaseprice > 0
#         GROUP BY
#             p.vendornumber, p.vendorname, p.brand,
#             p.description, p.purchaseprice, pp.price, pp.volume
#     ),
#     SalesSummary AS (
#         SELECT
#             vendorno,
#             brand,
#             SUM(salesquantity) AS TotalSalesQuantity,
#             SUM(salesdollars) AS TotalSalesDollars,
#             SUM(salesprice) AS TotalSalesPrice,
#             SUM(excisetax) AS TotalExciseTax
#         FROM sales
#         GROUP BY vendorno, brand
#     )
#     SELECT
#         ps.vendornumber,
#         ps.vendorname,
#         ps.brand,
#         ps.description,
#         ps.purchaseprice,
#         ps.ActualPrice,
#         ps.volume,
#         ps.TotalPurchaseQuantity,
#         ps.TotalPurchaseDollars,
#         ss.TotalSalesQuantity,
#         ss.TotalSalesDollars,
#         ss.TotalExciseTax,
#         fs.FreightCost
#     FROM PurchaseSummary ps
#     LEFT JOIN SalesSummary ss
#         ON ps.vendornumber = ss.vendorno
#         AND ps.brand = ss.brand
#     LEFT JOIN FreightSummary fs
#         ON ps.vendornumber = fs.vendornumber
#     ORDER BY ps.TotalPurchaseDollars DESC
#     """, conn)

#  return vendor_sales_summary


# def clean_data(df):
#   '''this function will clean the data'''
#   df['volume'] = df['volume'].astype(float)

#   df.fillna(0, inplace=True)

#   df['vendorname'] = df['vendorname'].str.strip()
#   df['description'] = df['description'].str.strip()

#   vendor_sales_summary['GrossProfit'] = vendor_sales_summary['TotalSalesDollars'] - vendor_sales_summary['TotalPurchaseDollars']
#   vendor_sales_summary['ProfitMargin'] = vendor_sales_summary['GrossProfit'] / vendor_sales_summary['TotalPurchaseDollars']*100
#   vendor_sales_summary['StockTurnover'] = vendor_sales_summary['TotalSalesQuantity'] / vendor_sales_summary['TotalPurchaseQuantity']
#   vendor_sales_summary['SalesPurchasesRatio'] = vendor_sales_summary['TotalSalesDollars'] / vendor_sales_summary['TotalPurchaseDollars']

#   return df

# if __name__ == "__main__":

#     #creating database connection
#   conn = sqlite3.connect(DB_PATH)

#   logging.info('Creating Vendor Summary Table.....')
#   summary_df = create_vendor_summary(conn)
#   logging.info(summary_df.head())

#   logging.info('Cleaning Data.....')
#   clean_df = clean_data(summary_df)
#   logging.info(clean_df.head())

#   logging.info('Ingesting Data.....')
#   ingest_db(clean_df, 'vendor_sales_summary')

#   logging.info('Closing Connection.....', conn)
#   logging.info('Completed')